<a href="https://colab.research.google.com/github/parthpranav2/ACCDB_to_CSV_converter/blob/main/ACCDB_to_CSV_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install mdbtools
!apt-get install -y mdbtools

# List tables
!mdb-tables your_file.accdb

# Export a specific table
!mdb-export your_file.accdb TableName > output.csv

In [ ]:
import pyodbc
print(pyodbc.version)

Single file conversion

In [ ]:
"""
ACCDB to CSV Converter using mdbtools on Google Colab
This script converts Microsoft Access (.accdb) files to CSV format
Works on Linux/Google Colab without Windows drivers!
"""

# Step 1: Install mdbtools
print("Installing mdbtools...")
!apt-get install -y mdbtools
print("✓ mdbtools installed\n")

# Step 2: Import libraries
import subprocess
import os
import pandas as pd
from google.colab import files
import zipfile

# Step 3: Upload your .accdb file
print("Please upload your .accdb file:")
uploaded = files.upload()

# Get the uploaded filename
accdb_filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded file: {accdb_filename}\n")

# Step 4: Get list of all tables
def get_table_list(accdb_file):
    """
    Get list of all tables in the Access database
    """
    try:
        result = subprocess.run(
            ['mdb-tables', '-1', accdb_file],
            capture_output=True,
            text=True,
            check=True
        )
        # Split by newline and filter out empty strings
        tables = [t.strip() for t in result.stdout.split('\n') if t.strip()]
        return tables
    except subprocess.CalledProcessError as e:
        print(f"✗ Error reading tables: {e}")
        return []

# Step 5: Export table to CSV
def export_table_to_csv(accdb_file, table_name, output_folder='output'):
    """
    Export a single table to CSV file
    """
    try:
        # Create output folder
        os.makedirs(output_folder, exist_ok=True)

        # Sanitize table name for filename
        safe_filename = table_name.replace(' ', '_').replace('/', '_')
        csv_filename = os.path.join(output_folder, f"{safe_filename}.csv")

        # Run mdb-export command
        with open(csv_filename, 'w') as f:
            result = subprocess.run(
                ['mdb-export', accdb_file, table_name],
                stdout=f,
                stderr=subprocess.PIPE,
                text=True,
                check=True
            )

        # Get row count
        df = pd.read_csv(csv_filename)
        row_count = len(df)

        print(f"✓ Exported: {table_name} → {csv_filename} ({row_count} rows)")
        return csv_filename

    except subprocess.CalledProcessError as e:
        print(f"✗ Error exporting {table_name}: {e.stderr}")
        return None
    except Exception as e:
        print(f"✗ Error exporting {table_name}: {e}")
        return None

# Step 6: Export table schema
def export_table_schema(accdb_file, table_name, output_folder='output'):
    """
    Export table schema (structure) to a text file
    """
    try:
        os.makedirs(output_folder, exist_ok=True)

        safe_filename = table_name.replace(' ', '_').replace('/', '_')
        schema_filename = os.path.join(output_folder, f"{safe_filename}_schema.txt")

        with open(schema_filename, 'w') as f:
            result = subprocess.run(
                ['mdb-schema', accdb_file, '-T', table_name],
                stdout=f,
                stderr=subprocess.PIPE,
                text=True
            )

        return schema_filename

    except Exception as e:
        print(f"⚠ Could not export schema for {table_name}: {e}")
        return None

# Step 7: Convert all tables
def convert_all_tables(accdb_file, include_schema=False):
    """
    Convert all tables in the database to CSV files
    """
    print("="*60)
    print("GETTING TABLE LIST")
    print("="*60 + "\n")

    tables = get_table_list(accdb_file)

    if not tables:
        print("✗ No tables found or error reading database")
        return []

    print(f"Found {len(tables)} table(s):\n")
    for i, table in enumerate(tables, 1):
        print(f"  {i}. {table}")

    print("\n" + "="*60)
    print("EXPORTING TABLES TO CSV")
    print("="*60 + "\n")

    csv_files = []
    schema_files = []

    for table in tables:
        csv_file = export_table_to_csv(accdb_file, table)
        if csv_file:
            csv_files.append(csv_file)

        if include_schema:
            schema_file = export_table_schema(accdb_file, table)
            if schema_file:
                schema_files.append(schema_file)

    print(f"\n✓ Successfully exported {len(csv_files)}/{len(tables)} tables")

    return csv_files, schema_files if include_schema else csv_files

# Step 8: Create ZIP archive
def create_zip_archive(files_list, zip_name='access_export.zip'):
    """
    Create a ZIP file containing all exported CSV files
    """
    if not files_list:
        return None

    with zipfile.ZipFile(zip_name, 'w') as zipf:
        for file in files_list:
            if os.path.exists(file):
                zipf.write(file, os.path.basename(file))

    print(f"\n✓ Created archive: {zip_name}")
    return zip_name

# Step 9: Download files
def download_files(csv_files, create_zip=True):
    """
    Download CSV files individually or as a ZIP
    """
    if not csv_files:
        print("No files to download")
        return

    print("\n" + "="*60)
    print("DOWNLOADING FILES")
    print("="*60 + "\n")

    if create_zip and len(csv_files) > 1:
        # Create and download ZIP
        zip_file = create_zip_archive(csv_files)
        if zip_file:
            files.download(zip_file)
            print(f"✓ Downloaded: {zip_file}")
    else:
        # Download individual files
        for csv_file in csv_files:
            if os.path.exists(csv_file):
                files.download(csv_file)
                print(f"✓ Downloaded: {os.path.basename(csv_file)}")

# Step 10: Preview data
def preview_data(csv_file, rows=5):
    """
    Preview the first few rows of a CSV file
    """
    try:
        df = pd.read_csv(csv_file)
        print(f"\nPreview of {os.path.basename(csv_file)}:")
        print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print("\nFirst {rows} rows:")
        print(df.head(rows))
        print("\n" + "-"*60 + "\n")
    except Exception as e:
        print(f"Could not preview {csv_file}: {e}")

# Step 11: Main execution
print("\n" + "="*60)
print("ACCDB TO CSV CONVERTER (using mdbtools)")
print("="*60 + "\n")

# Convert all tables
result = convert_all_tables(accdb_filename, include_schema=False)

if isinstance(result, tuple):
    csv_files, schema_files = result
else:
    csv_files = result
    schema_files = []

# Preview first table
if csv_files:
    print("\n" + "="*60)
    print("DATA PREVIEW")
    print("="*60)
    preview_data(csv_files[0], rows=5)

    # Ask user if they want to download
    download_choice = input("Download files? (yes/no): ").lower().strip()

    if download_choice in ['yes', 'y']:
        zip_choice = input("Download as ZIP? (yes/no): ").lower().strip()
        download_files(csv_files, create_zip=(zip_choice in ['yes', 'y']))
    else:
        print("\nFiles are saved in the 'output' folder")
        print("You can download them later using files.download()")
else:
    print("\n✗ No tables were successfully exported")

print("\n" + "="*60)
print("CONVERSION COMPLETE")
print("="*60)

Batch conversion